# Web attack detection

In [ ]:
from pathlib import Path
import hashlib
import re
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.utils import resample

SEED = 42
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "dataset" / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "preprocessed_with_labels"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "models"
BEST_MODEL_BUNDLE_PATH = ARTIFACT_DIR / "best_model_bundle.joblib"
ALL_MODELS_BUNDLE_PATH = ARTIFACT_DIR / "all_models_bundle.joblib"
CLASS_ORDER = ["Normal", "SQLi", "XSS", "OtherWebAttack"]
RUN_DEEP_MODEL = True
RUN_MULTIVIEW_DEEP_MODEL = True

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục dữ liệu: {DATA_ROOT}")
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy thư mục output đã chuẩn bị sẵn: {OUTPUT_DIR}")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
TF_AVAILABLE = True
print("TensorFlow available:", tf.__version__)

In [ ]:
TEXT_COLUMN_CANDIDATES = [
    "text", "sentence", "payload", "query", "request", "url", "content", "data", "input", "value"
]
LABEL_COLUMN_CANDIDATES = [
    "label", "class", "target", "attack", "type", "result", "is_attack"
]

CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
MULTISPACE_RE = re.compile(r"\s+")
HTTP_METHOD_RE = re.compile(r"^(get|post|put|delete|patch|head|options)\b", re.I)

NORMAL_VALUES = {"0", "normal", "benign", "safe", "clean", "false", "norm", "legitimate"}
ATTACK_VALUES = {"1", "attack", "malicious", "true", "anomaly", "anomalous"}
SQL_KEYWORDS = ["select", "union", "drop", "insert", "update", "delete", "sleep", "benchmark", "or 1=1", "waitfor"]
SCRIPT_KEYWORDS = ["<script", "javascript:", "onerror", "onload", "alert(", "document.cookie", "iframe", "svg"]
COMMAND_KEYWORDS = ["cmd", "whoami", "/bin/sh", "/bin/bash", "wget", "curl", "powershell", "cat /etc", "ls "]
TRAVERSAL_PATTERNS = ["../", "..\\", "%2e%2e%2f", "%2e%2e/", "..%2f", "/etc/passwd", "boot.ini", "win.ini"]


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8", errors="ignore")).hexdigest()


def normalize_text(value) -> str:
    value = "" if value is None else str(value)
    value = value.replace("\x00", " ")
    value = CONTROL_RE.sub(" ", value)
    value = value.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    value = MULTISPACE_RE.sub(" ", value).strip().lower()
    return value


def clean_column_name(name: str) -> str:
    name = str(name).replace("\ufeff", "").replace("\x00", "").strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
    return name or "unnamed"


def make_unique(columns):
    counts = {}
    unique = []
    for col in columns:
        count = counts.get(col, 0)
        unique.append(col if count == 0 else f"{col}_{count}")
        counts[col] = count + 1
    return unique


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = make_unique([clean_column_name(c) for c in df.columns])
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].map(lambda x: x if pd.isna(x) else str(x).replace("\x00", "").strip())
    return df


def read_csv_robust(path: Path) -> pd.DataFrame:
    attempts = [
        {"encoding": "utf-8-sig", "sep": ","},
        {"encoding": "utf-16", "sep": ","},
        {"encoding": "utf-16-le", "sep": ","},
        {"encoding": "utf-16-be", "sep": ","},
        {"encoding": "latin1", "sep": ","},
        {"encoding": "utf-8-sig", "sep": "\t"},
        {"encoding": "latin1", "sep": "\t"},
    ]
    last_error = None
    for attempt in attempts:
        try:
            df = pd.read_csv(path, encoding=attempt["encoding"], sep=attempt["sep"], engine="python")
            return clean_dataframe(df)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Không đọc được {path.name}: {last_error}")


def find_first(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def parse_binary_label(value, *, default=None):
    if pd.isna(value):
        return default
    value = normalize_text(value)
    if value in NORMAL_VALUES:
        return 0
    if value in ATTACK_VALUES:
        return 1
    if "normal" in value or "benign" in value:
        return 0
    if any(keyword in value for keyword in ["sqli", "sql", "xss", "attack", "inject", "traversal", "command"]):
        return 1
    return default


def extract_http_features(s):
    raw = "" if s is None else str(s)
    normalized = normalize_text(raw)
    length = max(len(normalized), 1)
    encoded_markers = raw.count('%') + raw.lower().count('&#')
    special_chars = sum(1 for ch in normalized if not ch.isalnum() and not ch.isspace())
    digits = sum(1 for ch in normalized if ch.isdigit())
    letters = sum(1 for ch in normalized if ch.isalpha())

    return {
        'text_length': len(normalized),
        'raw_length': len(raw),
        'length_delta': abs(len(raw) - len(normalized)),
        'num_question_mark': normalized.count('?'),
        'num_ampersand': normalized.count('&'),
        'num_equal': normalized.count('='),
        'num_quote': normalized.count("'") + normalized.count('"'),
        'num_semicolon': normalized.count(';'),
        'num_dash': normalized.count('-'),
        'num_angle_bracket': normalized.count('<') + normalized.count('>'),
        'num_parenthesis': normalized.count('(') + normalized.count(')'),
        'num_slash': normalized.count('/') + normalized.count('\\'),
        'num_percent': raw.count('%'),
        'num_special_chars': special_chars,
        'special_char_ratio': special_chars / length,
        'digit_ratio': digits / length,
        'letter_ratio': letters / length,
        'encoding_marker_count': encoded_markers,
        'encoding_marker_ratio': encoded_markers / max(len(raw), 1),
        'has_http_method': float(bool(HTTP_METHOD_RE.match(normalized))),
        'num_sql_keywords': sum(normalized.count(k) for k in SQL_KEYWORDS),
        'num_script_keywords': sum(normalized.count(k) for k in SCRIPT_KEYWORDS),
        'num_command_keywords': sum(normalized.count(k) for k in COMMAND_KEYWORDS),
        'num_traversal_patterns': sum(normalized.count(k) for k in TRAVERSAL_PATTERNS),
    }


NUMERIC_FEATURE_COLS = list(extract_http_features('sample').keys())
print(NUMERIC_FEATURE_COLS)


def summarize_rows(df: pd.DataFrame, name: str):
    print(f"{name}: {len(df):,} rows")
    if not df.empty:
        print(df["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))
        print()

In [ ]:
def load_csic_dataset(path: Path) -> pd.DataFrame:
    df = read_csv_robust(path)
    if df.empty:
        return pd.DataFrame(columns=["source", "raw_text", "label_binary", "label_multiclass", "label_origin"])

    label_col = df.columns[0]
    text_cols = [col for col in df.columns[1:] if not col.startswith("unnamed")]
    if not text_cols:
        raise ValueError("CSIC dataset không có cột nội dung để ghép request")

    records = []
    for _, row in df.iterrows():
        binary_label = parse_binary_label(row[label_col], default=None)
        if binary_label is None:
            continue

        parts = []
        for col in text_cols:
            value = row[col]
            if pd.isna(value):
                continue
            value = str(value).strip()
            if not value or value.lower() == "nan":
                continue
            parts.append(f"{col}={value}")

        raw_text = normalize_text(" | ".join(parts))
        if not raw_text:
            continue

        feature_map = extract_http_features(raw_text)
        records.append({
            "source": "CSIC",
            "raw_text": raw_text,
            "label_binary": binary_label,
            "label_multiclass": "Normal" if binary_label == 0 else "OtherWebAttack",
            "label_origin": "original",
            **feature_map,
        })

    return pd.DataFrame(records)


def load_payload_dataset(path: Path, source: str, attack_name: str) -> pd.DataFrame:
    df = read_csv_robust(path)
    text_col = find_first(df.columns, TEXT_COLUMN_CANDIDATES)
    label_col = find_first(df.columns, LABEL_COLUMN_CANDIDATES)

    if text_col is None:
        raise ValueError(f"{path.name}: không tìm thấy text column sau khi làm sạch header: {list(df.columns)}")

    records = []
    for _, row in df.iterrows():
        raw_text = normalize_text(row[text_col])
        if not raw_text or raw_text == "nan":
            continue

        binary_label = parse_binary_label(row[label_col], default=1) if label_col else 1
        feature_map = extract_http_features(raw_text)
        records.append({
            "source": source,
            "raw_text": raw_text,
            "label_binary": binary_label,
            "label_multiclass": "Normal" if binary_label == 0 else attack_name,
            "label_origin": "original",
            **feature_map,
        })

    return pd.DataFrame(records)


def build_master_dataset() -> pd.DataFrame:
    manifest = [
        {"kind": "csic", "path": DATA_ROOT / "http-CSIC-2010" / "csic_database.csv"},
        {"kind": "payload", "path": DATA_ROOT / "sqlinjectionextend" / "sqli-extended.csv", "source": "SQLI_EXTENDED", "attack_name": "SQLi"},
        {"kind": "payload", "path": DATA_ROOT / "sqli" / "sqli.csv", "source": "SQLI", "attack_name": "SQLi"},
        {"kind": "payload", "path": DATA_ROOT / "sqli" / "sqliv2.csv", "source": "SQLI_V2", "attack_name": "SQLi"},
        {"kind": "payload", "path": DATA_ROOT / "sqli" / "SQLiV3.csv", "source": "SQLI_V3", "attack_name": "SQLi"},
        {"kind": "payload", "path": DATA_ROOT / "xss" / "XSS_dataset.csv", "source": "XSS", "attack_name": "XSS"},
    ]

    frames = []
    for item in manifest:
        path = item["path"]
        if not path.exists():
            print("Bỏ qua file không tồn tại:", path)
            continue
        frame = load_csic_dataset(path) if item["kind"] == "csic" else load_payload_dataset(path, item["source"], item["attack_name"])
        summarize_rows(frame, path.name)
        frames.append(frame)

    if not frames:
        raise RuntimeError("Không có dataset nào được load thành công")

    data = pd.concat(frames, ignore_index=True)
    data["normalized_text"] = data["raw_text"].map(normalize_text)
    data = data[data["normalized_text"].str.len() > 0].copy()
    data["text_hash"] = data["normalized_text"].map(sha256_text)
    return data

## EDA 1 - Phân bố lớp ban đầu
Nhìn nhanh phân bố 4 lớp và độ dài text trước khi cleaning và split.

In [ ]:
data = build_master_dataset()
print("Tổng số dòng sau load:", f"{len(data):,}")
print()
print(data["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))
print()
print(data["source"].value_counts())

overview = (
    data.groupby("label_multiclass")
    .agg(rows=("label_multiclass", "size"), avg_len=("text_length", "mean"), median_len=("text_length", "median"))
    .reindex(CLASS_ORDER)
)
print()
print(overview)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
data["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0).plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518", "#E45756", "#72B7B2"])
axes[0].set_title("Class distribution before cleaning")
axes[0].tick_params(axis="x", rotation=20)

data.boxplot(column="text_length", by="label_multiclass", ax=axes[1], grid=False)
axes[1].set_title("Text length by class")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Characters")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Cleaning và lưu dữ liệu tiền xử lý
Notebook chỉ sinh file trong `dataset/preprocessed_with_labels`.

In [ ]:
conflict_hashes = (
    data.groupby("text_hash")["label_multiclass"]
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

before_conflict = len(data)
data = data[~data["text_hash"].isin(conflict_hashes)].copy()
after_conflict = len(data)

before_dedup = len(data)
data = data.drop_duplicates(subset="text_hash", keep="first").reset_index(drop=True)
after_dedup = len(data)

print("Rows after cleaning:", f"{len(data):,}")
print("Removed by multiclass conflict:", before_conflict - after_conflict)
print("Removed by dedup:", before_dedup - after_dedup)
print()
print(data["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))

standardized_path = OUTPUT_DIR / "standardized_multiclass_dataset.csv"
data.to_csv(standardized_path, index=False)
print("Saved:", standardized_path)

## Train / validation / test split
Split được thực hiện sau cleaning để hạn chế leakage do duplicate text.

In [ ]:
if data["label_multiclass"].nunique() < 4:
    raise RuntimeError("Notebook mong đợi đúng 4 lớp bám sát dataset gốc")

train_df, temp_df = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data["label_multiclass"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_multiclass"],
)

for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, len(frame))
    print(frame["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))
    print()

split_overlap = {
    "train_val": len(set(train_df["text_hash"]) & set(val_df["text_hash"])),
    "train_test": len(set(train_df["text_hash"]) & set(test_df["text_hash"])),
    "val_test": len(set(val_df["text_hash"]) & set(test_df["text_hash"])),
}
print("Hash overlap:", split_overlap)
assert split_overlap == {"train_val": 0, "train_test": 0, "val_test": 0}

train_df.to_csv(OUTPUT_DIR / "train_multiclass.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "validation_multiclass.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test_multiclass.csv", index=False)
print("Saved split files to:", OUTPUT_DIR)

## EDA 2 - Quan sát imbalance trên train split
Ta balance train split bằng oversampling trước khi train model.

In [ ]:
def make_balanced_training_frame(frame: pd.DataFrame, label_col: str = "label_multiclass") -> pd.DataFrame:
    groups = []
    class_sizes = frame[label_col].value_counts()
    target_size = class_sizes.max()

    for label, group in frame.groupby(label_col):
        if len(group) < target_size:
            sampled = resample(group, replace=True, n_samples=target_size, random_state=SEED)
            groups.append(sampled)
        else:
            groups.append(group)

    balanced = pd.concat(groups, ignore_index=True)
    return balanced.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


train_balanced_df = make_balanced_training_frame(train_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train_df["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0).plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Train distribution before balancing")
axes[0].tick_params(axis="x", rotation=20)

train_balanced_df["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0).plot(kind="bar", ax=axes[1], color="#54A24B")
axes[1].set_title("Train distribution after oversampling")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

print("Before balancing:")
print(train_df["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))
print()
print("After balancing:")
print(train_balanced_df["label_multiclass"].value_counts().reindex(CLASS_ORDER, fill_value=0))

train_balanced_df.to_csv(OUTPUT_DIR / "train_multiclass_balanced.csv", index=False)
print("Saved:", OUTPUT_DIR / "train_multiclass_balanced.csv")

## Chuẩn bị helper cho đánh giá model
Ta dùng validation để chọn model tốt nhất và test set chỉ report cuối.

In [ ]:
def evaluate_frame(model, frame: pd.DataFrame, split_name: str, *, input_kind: str = 'text') -> dict:
    if input_kind == 'text':
        X = frame["normalized_text"]
    elif input_kind == 'numeric':
        X = frame[NUMERIC_FEATURE_COLS].fillna(0)
    elif input_kind == 'multiview':
        X = frame[["normalized_text"] + NUMERIC_FEATURE_COLS].copy()
    else:
        raise ValueError(input_kind)

    y_true = frame["label_multiclass"]
    y_pred = model.predict(X)
    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }


def evaluate_deep_predictions(y_true, y_pred, split_name: str, model_name: str, train_time_sec=None, inference_time_ms=None):
    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "train_time_sec": None if train_time_sec is None else float(train_time_sec),
        "inference_time_ms": None if inference_time_ms is None else float(inference_time_ms),
    }


def plot_train_valid_metrics(model_name: str, train_metrics: dict, valid_metrics: dict):
    metric_names = ['accuracy', 'f1_macro', 'f1_weighted']
    labels = ['train', 'validation']
    plt.figure(figsize=(8, 4.5))
    for metric_name in metric_names:
        plt.plot(labels, [train_metrics[metric_name], valid_metrics[metric_name]], marker='o', linewidth=2, label=metric_name)
    plt.ylim(0, 1.0)
    plt.ylabel('Score')
    plt.title(f'Train vs validation metrics - {model_name}')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_keras_history(history, model_name: str):
    history_df = pd.DataFrame(history.history)
    history_df['epoch'] = np.arange(1, len(history_df) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(history_df['epoch'], history_df['loss'], marker='o', label='train_loss')
    if 'val_loss' in history_df:
        axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val_loss')
    axes[0].set_title(f'Loss by epoch - {model_name}')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    if 'accuracy' in history_df:
        axes[1].plot(history_df['epoch'], history_df['accuracy'], marker='o', label='train_accuracy')
    if 'val_accuracy' in history_df:
        axes[1].plot(history_df['epoch'], history_df['val_accuracy'], marker='o', label='val_accuracy')
    axes[1].set_title(f'Accuracy by epoch - {model_name}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.0)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Train tất cả model chính từ notebook gốc
Bao gồm 4 model classical và 2 model deep learning tương ứng với pipeline gốc.

In [ ]:
validation_results = []
trained_models = {}

# 1) TF-IDF + Logistic Regression
model_name = 'logistic_regression_tfidf'
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=100000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
])
start = time.perf_counter()
lr_pipeline.fit(train_balanced_df['normalized_text'], train_balanced_df['label_multiclass'])
train_time = time.perf_counter() - start
train_metrics = evaluate_frame(lr_pipeline, train_balanced_df, 'train', input_kind='text')
metrics = evaluate_frame(lr_pipeline, val_df, 'validation', input_kind='text')
metrics.update({'model': model_name, 'train_time_sec': train_time})
validation_results.append(metrics)
trained_models[model_name] = {'model': lr_pipeline, 'input_kind': 'text'}
plot_train_valid_metrics(model_name, train_metrics, metrics)

# 2) TF-IDF + Linear SVC
model_name = 'linear_svm_tfidf'
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=120000)),
    ('clf', LinearSVC(class_weight='balanced', random_state=SEED))
])
start = time.perf_counter()
svm_pipeline.fit(train_balanced_df['normalized_text'], train_balanced_df['label_multiclass'])
train_time = time.perf_counter() - start
train_metrics = evaluate_frame(svm_pipeline, train_balanced_df, 'train', input_kind='text')
metrics = evaluate_frame(svm_pipeline, val_df, 'validation', input_kind='text')
metrics.update({'model': model_name, 'train_time_sec': train_time})
validation_results.append(metrics)
trained_models[model_name] = {'model': svm_pipeline, 'input_kind': 'text'}
plot_train_valid_metrics(model_name, train_metrics, metrics)

# 3) Statistical features + Random Forest
model_name = 'statistical_features_random_forest'
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, max_depth=None, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1))
])
start = time.perf_counter()
rf_pipeline.fit(train_balanced_df[NUMERIC_FEATURE_COLS].fillna(0), train_balanced_df['label_multiclass'])
train_time = time.perf_counter() - start
train_metrics = evaluate_frame(rf_pipeline, train_balanced_df, 'train', input_kind='numeric')
metrics = evaluate_frame(rf_pipeline, val_df, 'validation', input_kind='numeric')
metrics.update({'model': model_name, 'train_time_sec': train_time})
validation_results.append(metrics)
trained_models[model_name] = {'model': rf_pipeline, 'input_kind': 'numeric'}
plot_train_valid_metrics(model_name, train_metrics, metrics)

# 4) Multi-view TF-IDF + stats + Logistic Regression
model_name = 'multiview_tfidf_stats_logistic_regression'
preprocess = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=100000), 'normalized_text'),
        ('num', StandardScaler(), NUMERIC_FEATURE_COLS),
    ],
    remainder='drop'
)
mv_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))
])
start = time.perf_counter()
mv_pipeline.fit(train_balanced_df[['normalized_text'] + NUMERIC_FEATURE_COLS], train_balanced_df['label_multiclass'])
train_time = time.perf_counter() - start
train_metrics = evaluate_frame(mv_pipeline, train_balanced_df, 'train', input_kind='multiview')
metrics = evaluate_frame(mv_pipeline, val_df, 'validation', input_kind='multiview')
metrics.update({'model': model_name, 'train_time_sec': train_time})
validation_results.append(metrics)
trained_models[model_name] = {'model': mv_pipeline, 'input_kind': 'multiview'}
plot_train_valid_metrics(model_name, train_metrics, metrics)

validation_table = pd.DataFrame(validation_results)

In [ ]:
if RUN_DEEP_MODEL and TF_AVAILABLE:
    model_name = 'cnn_bilstm_text'
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(train_balanced_df["label_multiclass"].astype(str))
    y_val_enc = label_encoder.transform(val_df["label_multiclass"].astype(str))
    class_names = label_encoder.classes_.tolist()
    num_classes = len(class_names)

    max_tokens = 50000
    sequence_length = 256
    vectorizer = layers.TextVectorization(max_tokens=max_tokens, output_mode='int', output_sequence_length=sequence_length, standardize=None)
    vectorizer.adapt(train_balanced_df["normalized_text"].astype(str).values)

    text_input = keras.Input(shape=(1,), dtype=tf.string, name='text')
    x = vectorizer(text_input)
    x = layers.Embedding(input_dim=max_tokens, output_dim=64)(x)
    x = layers.Conv1D(filters=96, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.25)(x)
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model_text = keras.Model(inputs=text_input, outputs=output)
    model_text.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

    start = time.perf_counter()
    history = model_text.fit(
        train_balanced_df["normalized_text"].astype(str).values,
        y_train_enc,
        validation_data=(val_df["normalized_text"].astype(str).values, y_val_enc),
        epochs=20,
        batch_size=128,
        callbacks=callbacks,
        verbose=1,
    )
    train_time = time.perf_counter() - start
    plot_keras_history(history, model_name)

    train_pred_raw = model_text.predict(train_balanced_df["normalized_text"].astype(str).values, batch_size=128, verbose=0)
    train_pred_enc = np.argmax(train_pred_raw, axis=1)
    train_pred = label_encoder.inverse_transform(train_pred_enc)
    train_true = train_balanced_df["label_multiclass"].astype(str).values
    train_metrics = evaluate_deep_predictions(train_true, train_pred, 'train', model_name)

    start = time.perf_counter()
    pred_raw = model_text.predict(val_df["normalized_text"].astype(str).values, batch_size=128, verbose=0)
    inference_ms = (time.perf_counter() - start) / max(len(val_df), 1) * 1000
    y_pred_enc = np.argmax(pred_raw, axis=1)
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    y_true = val_df["label_multiclass"].astype(str).values

    metrics = evaluate_deep_predictions(y_true, y_pred, 'validation', model_name, train_time, inference_ms)
    plot_train_valid_metrics(model_name, train_metrics, metrics)
    validation_results.append(metrics)
    trained_models[model_name] = {'model': model_text, 'input_kind': 'keras_text', 'label_encoder': label_encoder}
else:
    print('Skipped cnn_bilstm_text')

In [ ]:
if RUN_MULTIVIEW_DEEP_MODEL and TF_AVAILABLE:
    model_name = 'multiview_cnn_bilstm_text_stats'
    label_encoder_mv = LabelEncoder()
    y_train_enc = label_encoder_mv.fit_transform(train_balanced_df["label_multiclass"].astype(str))
    y_val_enc = label_encoder_mv.transform(val_df["label_multiclass"].astype(str))
    class_names = label_encoder_mv.classes_.tolist()
    num_classes = len(class_names)

    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(train_balanced_df[NUMERIC_FEATURE_COLS].fillna(0))
    X_val_num = scaler.transform(val_df[NUMERIC_FEATURE_COLS].fillna(0))

    max_tokens = 50000
    sequence_length = 256
    vectorizer_mv = layers.TextVectorization(max_tokens=max_tokens, output_mode='int', output_sequence_length=sequence_length, standardize=None)
    vectorizer_mv.adapt(train_balanced_df["normalized_text"].astype(str).values)

    text_input = keras.Input(shape=(1,), dtype=tf.string, name='text')
    text_branch = vectorizer_mv(text_input)
    text_branch = layers.Embedding(input_dim=max_tokens, output_dim=64)(text_branch)
    text_branch = layers.Conv1D(filters=96, kernel_size=5, activation='relu', padding='same')(text_branch)
    text_branch = layers.MaxPooling1D(pool_size=2)(text_branch)
    text_branch = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(text_branch)
    text_branch = layers.Dropout(0.35)(text_branch)

    stats_input = keras.Input(shape=(len(NUMERIC_FEATURE_COLS),), dtype=tf.float32, name='stats')
    stats_branch = layers.Dense(64, activation='relu')(stats_input)
    stats_branch = layers.Dropout(0.25)(stats_branch)
    stats_branch = layers.Dense(32, activation='relu')(stats_branch)

    x = layers.Concatenate()([text_branch, stats_branch])
    x = layers.Dense(96, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model_mv = keras.Model(inputs=[text_input, stats_input], outputs=output)
    model_mv.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

    train_inputs = (train_balanced_df["normalized_text"].astype(str).values, X_train_num.astype('float32'))
    val_inputs = (val_df["normalized_text"].astype(str).values, X_val_num.astype('float32'))

    start = time.perf_counter()
    history = model_mv.fit(
        train_inputs,
        y_train_enc,
        validation_data=(val_inputs, y_val_enc),
        epochs=20,
        batch_size=128,
        callbacks=callbacks,
        verbose=1,
    )
    train_time = time.perf_counter() - start
    plot_keras_history(history, model_name)

    train_pred_raw = model_mv.predict(train_inputs, batch_size=128, verbose=0)
    train_pred_enc = np.argmax(train_pred_raw, axis=1)
    train_pred = label_encoder_mv.inverse_transform(train_pred_enc)
    train_true = train_balanced_df["label_multiclass"].astype(str).values
    train_metrics = evaluate_deep_predictions(train_true, train_pred, 'train', model_name)

    start = time.perf_counter()
    pred_raw = model_mv.predict(val_inputs, batch_size=128, verbose=0)
    inference_ms = (time.perf_counter() - start) / max(len(val_df), 1) * 1000
    y_pred_enc = np.argmax(pred_raw, axis=1)
    y_pred = label_encoder_mv.inverse_transform(y_pred_enc)
    y_true = val_df["label_multiclass"].astype(str).values

    metrics = evaluate_deep_predictions(y_true, y_pred, 'validation', model_name, train_time, inference_ms)
    plot_train_valid_metrics(model_name, train_metrics, metrics)
    validation_results.append(metrics)
    trained_models[model_name] = {'model': model_mv, 'input_kind': 'keras_multiview', 'label_encoder': label_encoder_mv, 'scaler': scaler}
else:
    print('Skipped multiview_cnn_bilstm_text_stats')

In [ ]:
validation_table = pd.DataFrame(validation_results).sort_values(["f1_macro", "accuracy"], ascending=False).reset_index(drop=True)
print(validation_table)
print()
print("Validation table chỉ để tham khảo. Tất cả model sẽ được lưu và đánh giá riêng trên test set.")

## Đánh giá toàn bộ model trên test set
Validation chỉ để tham khảo. Tất cả model sẽ được fit lại trên `train + validation`, lưu artifact riêng, rồi đánh giá trên test set để người dùng tự so sánh.

In [ ]:
def rebalance_after_concat(frame: pd.DataFrame) -> pd.DataFrame:
    return make_balanced_training_frame(frame, label_col="label_multiclass")


def fit_and_predict_model(model_name: str, model_info: dict, train_frame: pd.DataFrame, test_frame: pd.DataFrame):
    input_kind = model_info['input_kind']
    model = model_info['model']
    y_true = test_frame['label_multiclass']
    extra_bundle = {}

    if input_kind == 'text':
        model.fit(train_frame['normalized_text'], train_frame['label_multiclass'])
        y_pred = model.predict(test_frame['normalized_text'])
    elif input_kind == 'numeric':
        model.fit(train_frame[NUMERIC_FEATURE_COLS].fillna(0), train_frame['label_multiclass'])
        y_pred = model.predict(test_frame[NUMERIC_FEATURE_COLS].fillna(0))
    elif input_kind == 'multiview':
        model.fit(train_frame[['normalized_text'] + NUMERIC_FEATURE_COLS], train_frame['label_multiclass'])
        y_pred = model.predict(test_frame[['normalized_text'] + NUMERIC_FEATURE_COLS])
    elif input_kind == 'keras_text':
        label_encoder = model_info['label_encoder']
        y_train_enc = label_encoder.fit_transform(train_frame['label_multiclass'].astype(str))
        y_test_enc = label_encoder.transform(test_frame['label_multiclass'].astype(str))
        vectorizer = model.layers[1]
        vectorizer.adapt(train_frame['normalized_text'].astype(str).values)
        callbacks = [keras.callbacks.EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)]
        model.fit(train_frame['normalized_text'].astype(str).values, y_train_enc, epochs=8, batch_size=128, callbacks=callbacks, verbose=1)
        pred_raw = model.predict(test_frame['normalized_text'].astype(str).values, batch_size=128, verbose=0)
        y_pred_enc = np.argmax(pred_raw, axis=1)
        y_true = label_encoder.inverse_transform(y_test_enc)
        y_pred = label_encoder.inverse_transform(y_pred_enc)
        extra_bundle['label_encoder'] = label_encoder
    elif input_kind == 'keras_multiview':
        label_encoder = model_info['label_encoder']
        y_train_enc = label_encoder.fit_transform(train_frame['label_multiclass'].astype(str))
        y_test_enc = label_encoder.transform(test_frame['label_multiclass'].astype(str))
        scaler = StandardScaler()
        X_train_num = scaler.fit_transform(train_frame[NUMERIC_FEATURE_COLS].fillna(0))
        X_test_num = scaler.transform(test_frame[NUMERIC_FEATURE_COLS].fillna(0))
        text_vectorizer = model.get_layer(index=1)
        text_vectorizer.adapt(train_frame['normalized_text'].astype(str).values)
        callbacks = [keras.callbacks.EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)]
        model.fit((train_frame['normalized_text'].astype(str).values, X_train_num.astype('float32')), y_train_enc, epochs=8, batch_size=128, callbacks=callbacks, verbose=1)
        pred_raw = model.predict((test_frame['normalized_text'].astype(str).values, X_test_num.astype('float32')), batch_size=128, verbose=0)
        y_pred_enc = np.argmax(pred_raw, axis=1)
        y_true = label_encoder.inverse_transform(y_test_enc)
        y_pred = label_encoder.inverse_transform(y_pred_enc)
        extra_bundle['label_encoder'] = label_encoder
        extra_bundle['scaler'] = scaler
    else:
        raise ValueError(input_kind)

    result = {
        'model': model_name,
        'input_kind': input_kind,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
    }
    return model, np.asarray(y_true), np.asarray(y_pred), result, extra_bundle


def save_model_bundle(model_name: str, model_info: dict, fitted_model, extra_bundle: dict):
    bundle = {
        'model_name': model_name,
        'input_kind': model_info['input_kind'],
        'text_column': 'normalized_text',
        'label_column': 'label_multiclass',
        'numeric_feature_cols': NUMERIC_FEATURE_COLS,
        'class_order': CLASS_ORDER,
    }

    keras_model_path = None
    if model_info['input_kind'] in {'text', 'numeric', 'multiview'}:
        bundle['model'] = fitted_model
    else:
        keras_model_path = ARTIFACT_DIR / f'{model_name}.keras'
        fitted_model.save(keras_model_path)
        bundle['model_path'] = str(keras_model_path)
        bundle.update(extra_bundle)

    bundle_path = ARTIFACT_DIR / f'{model_name}_bundle.joblib'
    joblib.dump(bundle, bundle_path)
    return bundle, bundle_path, keras_model_path


def load_model_bundle(bundle_path):
    loaded_bundle = joblib.load(bundle_path)
    if loaded_bundle['input_kind'].startswith('keras'):
        if not TF_AVAILABLE:
            raise RuntimeError('TensorFlow chưa khả dụng nên không thể load Keras model đã lưu.')
        loaded_bundle['model'] = keras.models.load_model(loaded_bundle['model_path'])
    return loaded_bundle


trainval_df = pd.concat([train_df, val_df], ignore_index=True)
trainval_balanced_df = rebalance_after_concat(trainval_df)
test_results = []
saved_model_registry = {}
all_model_bundles = {}
test_predictions = {}

for model_name, model_info in trained_models.items():
    print(f'=== Running on test set: {model_name} ===')
    fitted_model, y_true, y_pred, result, extra_bundle = fit_and_predict_model(model_name, model_info, trainval_balanced_df, test_df)
    test_results.append(result)
    test_predictions[model_name] = {'y_true': y_true, 'y_pred': y_pred}

    bundle, bundle_path, keras_model_path = save_model_bundle(model_name, model_info, fitted_model, extra_bundle)
    all_model_bundles[model_name] = bundle
    saved_model_registry[model_name] = {
        'bundle_path': str(bundle_path),
        'keras_model_path': None if keras_model_path is None else str(keras_model_path),
        'input_kind': model_info['input_kind'],
    }

    print(pd.DataFrame([result]))
    print(classification_report(y_true, y_pred, labels=CLASS_ORDER, zero_division=0))
    print()

test_results_table = pd.DataFrame(test_results).sort_values(['f1_macro', 'accuracy'], ascending=False).reset_index(drop=True)
print('=== Test results comparison ===')
print(test_results_table)

comparison_ax = test_results_table.set_index('model')[['accuracy', 'f1_macro', 'f1_weighted']].plot(kind='bar', figsize=(14, 6))
comparison_ax.set_title('Model comparison on test set')
comparison_ax.set_ylabel('Score')
comparison_ax.set_xlabel('Model')
comparison_ax.tick_params(axis='x', rotation=20)
comparison_ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

for model_name, pred_info in test_predictions.items():
    cm = confusion_matrix(pred_info['y_true'], pred_info['y_pred'], labels=CLASS_ORDER)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER).plot(cmap='Blues', xticks_rotation=20)
    plt.title(f'Test confusion matrix - {model_name}')
    plt.show()

joblib.dump(all_model_bundles, ALL_MODELS_BUNDLE_PATH)
if test_results_table.empty:
    raise RuntimeError('Không có model nào được đánh giá trên test set.')

def load_saved_model_bundle(model_name: str):
    bundle_path = ARTIFACT_DIR / f'{model_name}_bundle.joblib'
    return load_model_bundle(bundle_path)


def load_all_model_bundles(bundle_path=ALL_MODELS_BUNDLE_PATH):
    loaded = joblib.load(bundle_path)
    restored = {}
    for model_name, bundle in loaded.items():
        if bundle['input_kind'].startswith('keras'):
            if not TF_AVAILABLE:
                raise RuntimeError('TensorFlow chưa khả dụng nên không thể load Keras model đã lưu.')
            bundle = dict(bundle)
            bundle['model'] = keras.models.load_model(bundle['model_path'])
        restored[model_name] = bundle
    return restored

print('Saved all model bundles to:', ALL_MODELS_BUNDLE_PATH)
print('Per-model artifact registry:')
print(pd.DataFrame(saved_model_registry).T)